# Phương pháp #4 — LAD (L1) + Laplacian regularization

$$\hat x=\arg\min_x\ \sum_{i\in\mathcal M}|x_i-y_i|\;+\;\lambda\,x^\top Lx,
\qquad x^\top Lx=\sum_{(i,j)\in E}w_{ij}(x_i-x_j)^2$$

Notebook chạy được từ đầu đến cuối (`Run All`) từ bất kỳ thư mục con nào của repo. Cần: `numpy scipy networkx matplotlib cvxpy`. `public_cases.json` được tìm ở thư mục gốc repo; hình lưu vào `figures/method_4/`, kết quả (`xhat_4.npy`, `method_4.csv`) lưu vào `results/method_4/`. Solver cũng có ở module `src/method_4/method4_lad.py`.

| Mục | Nội dung |
|---|---|
| 1 | Dữ liệu nhóm (giống Mục III–IV của notebook gốc) |
| 2 | Code chung của nhóm (hợp đồng) |
| 3 | Solver: `solve_lad` (CVXPY), `solve_lad_irls` (tự cài), `solve_tikhonov` (baseline) |
| 4 | Kiểm tra đúng đắn: CVXPY vs IRLS, KKT, ngưỡng $\lambda_0$, tính (không) duy nhất |
| 5 | Đường hội tụ IRLS |
| 6 | Thực nghiệm giả lập 20 seed |
| 7 | Dữ liệu thật: chọn $\lambda$ bằng CV, sinh `xhat_4.npy`, hình |
| 8 | Kiểm thử trên nhiều bộ đề của `public_cases.json` |
| 9 | Dòng kết quả + thuyết minh |

## 1. Dữ liệu nhóm

In [ ]:
# Giống Mục III notebook gốc — THAY bằng MSSV thật của nhóm (cả nhóm dùng CÙNG danh sách).
NHOM_MSSV = ["20260001", "20260002", "20260003", "20260004",
             "20260005", "20260006", "20260007"]
MSSV_DAI_DIEN = NHOM_MSSV[0]

In [ ]:
import json, hashlib, time
from pathlib import Path
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# --- copy nguyên văn từ Mục IV notebook gốc ---
def assign_mssv_to_case(student_id, cases):
    sid = str(student_id).strip()
    if sid in cases:
        return sid
    case_ids = sorted(cases, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
    k = len(case_ids)
    if sid.isdecimal():
        index = int(sid) % k
    else:
        index = int.from_bytes(hashlib.sha256(sid.encode("utf-8")).digest(), "big") % k
    return case_ids[index]

def case_to_graph(case):
    G = nx.Graph()
    for i, xy in enumerate(case["positions"]):
        G.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in case["edges"]:
        G.add_edge(int(u), int(v), weight=float(w))
    y = np.asarray(case["y"], dtype=float)
    assert nx.is_connected(G) and len(y) == G.number_of_nodes()
    return G, y

# Thư mục gốc repo = thư mục chứa public_cases.json (chạy được từ mọi thư mục con)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "public_cases.json").exists())
FIG_DIR = ROOT / "figures" / "method_4"; FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = ROOT / "results" / "method_4"; RES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = ROOT / "public_cases.json"
CASES = json.loads(DATA_PATH.read_text(encoding="utf-8"))["cases"]
CASE_ID = assign_mssv_to_case(MSSV_DAI_DIEN, CASES)
G, y = case_to_graph(CASES[CASE_ID])
positions = nx.get_node_attributes(G, "pos")
print("MSSV đại diện:", MSSV_DAI_DIEN, "| bộ đề:", CASE_ID)
print(f"n = {G.number_of_nodes()}, |E| = {G.number_of_edges()}")

## 2. Code chung của nhóm (hợp đồng — không sửa)

In [ ]:
n = len(y)
L = nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()

# (a) Dữ liệu giả lập: biết x thật, dùng để đo RMSE
def make_synthetic(G, y, seed):
    rng = np.random.default_rng(seed)
    lam, U = np.linalg.eigh(L)
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam[:5]))
    x = np.median(y) + np.std(y) * x / np.std(x)
    sigma = 0.1 * np.std(y)
    y_syn = x + rng.normal(0, sigma, n)
    idx = rng.choice(n, 3, replace=False)                      # 3 outlier
    y_syn[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, y_syn

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))

# (b) Chọn tham số trên dữ liệu thật: che 1/5 số đỉnh, dự đoán lại
def cv_score(solve, y, params, k=5, seed=0):
    folds = np.random.default_rng(seed).permutation(n) % k
    err = []
    for f in range(k):
        mask = folds != f                                      # True = được quan sát
        x = solve(G, y, mask=mask, **params)
        err.append(np.abs(x[~mask] - y[~mask]))
    return np.median(np.concatenate(err))                      # median để không bị outlier kéo lệch

## 3. Solver

* **`solve_lad`** — nghiệm chính xác. Bài toán là QP lồi; CVXPY + Clarabel. Viết $x^\top Lx=\|Dx\|^2$ với $D=\sqrt W B$ (tránh lỗi PSD của `quad_form`), và viết $m\odot x-m\odot y$ để bài toán là **DPP** (biên dịch 1 lần, mỗi lần giải ~2 ms).
* **`solve_lad_irls`** — tự cài để đối chiếu. Majorize $|r|\le \frac{r^2}{2|r_k|}+\frac{|r_k|}{2}$ ⇒ mỗi bước giải hệ tuyến tính
  $(W_k+2\lambda L)x=W_k y,\ W_k=\mathrm{diag}\!\big(m_i/\max(|x_i^{(k)}-y_i|,\varepsilon)\big)$; $\varepsilon$ giảm dần $10^{-1}\to10^{-10}$. Hàm mục tiêu giảm đơn điệu (MM).
* **`solve_tikhonov`** — baseline L2 $(M+\lambda L)x=My$.

In [ ]:
import numpy as np
import networkx as nx
import cvxpy as cp


# ----------------------------------------------------------------------------
# Tiện ích đồ thị
# ----------------------------------------------------------------------------
def laplacian(G):
    n = G.number_of_nodes()
    return nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()


def weighted_incidence(G):
    """D sao cho x^T L x = ||D x||^2  (D = sqrt(W) B, B là ma trận liên thuộc cạnh-đỉnh)."""
    n = G.number_of_nodes()
    edges = list(G.edges(data="weight"))
    D = np.zeros((len(edges), n))
    for k, (u, v, w) in enumerate(edges):
        s = np.sqrt(w)
        D[k, u], D[k, v] = s, -s
    return D


def _mask(n, mask):
    return np.ones(n, bool) if mask is None else np.asarray(mask, bool)


def lad_objective(x, y, L, lam, mask=None):
    m = _mask(len(y), mask)
    return float(np.abs(x - y)[m].sum() + lam * x @ L @ x)


def lambda0(G, y):
    """Với lam < lambda0, x = y thỏa KKT  =>  nghiệm LAD chính là y (không khử nhiễu)."""
    return 1.0 / (2.0 * np.abs(laplacian(G) @ y).max())


# ----------------------------------------------------------------------------
# Solver 1: CVXPY (DPP – biên dịch 1 lần cho mỗi đồ thị, sau đó chỉ đổi tham số)
# ----------------------------------------------------------------------------
_CACHE = {}


def _get_problem(G):
    # key theo nội dung đồ thị (không dùng id(G): id có thể bị tái sử dụng)
    key = (G.number_of_nodes(), tuple(sorted((min(u, v), max(u, v), round(w, 12))
                                         for u, v, w in G.edges(data="weight"))))
    if key not in _CACHE:
        n = G.number_of_nodes()
        D = weighted_incidence(G)
        x = cp.Variable(n)
        my_p = cp.Parameter(n)                 # m * y
        m_p = cp.Parameter(n, nonneg=True)     # 1 = quan sát, 0 = bị che
        lam_p = cp.Parameter(nonneg=True)
        # viết m*x - m*y (thay vì m*(x-y)) để bài toán là DPP -> chỉ biên dịch 1 lần
        obj = cp.norm1(cp.multiply(m_p, x) - my_p) + lam_p * cp.sum_squares(D @ x)
        _CACHE[key] = (cp.Problem(cp.Minimize(obj)), x, my_p, m_p, lam_p)
    return _CACHE[key]


def solve_lad(G, y, mask=None, lam=1.0, solver="CLARABEL"):
    prob, x, my_p, m_p, lam_p = _get_problem(G)
    m = _mask(len(y), mask).astype(float)
    m_p.value = m
    my_p.value = m * np.asarray(y, float)
    lam_p.value = float(lam)
    prob.solve(solver=solver)
    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"CVXPY status = {prob.status}")
    return np.asarray(x.value, float).ravel()


# ----------------------------------------------------------------------------
# Solver 2: IRLS (majorize-minimize)
#   |r| <= r^2 / (2|r_k|) + |r_k|/2   =>   mỗi bước giải  (W + 2 lam L) x = W y,
#   W = diag(m_i / max(|r_i|, eps)).  eps giảm dần để tiến tới bài toán L1 gốc.
# ----------------------------------------------------------------------------
def solve_lad_irls(G, y, mask=None, lam=1.0, eps0=1e-1, eps_min=1e-10,
                   max_iter=3000, tol_stage=1e-7, tol=1e-12, return_history=False):
    y = np.asarray(y, float)
    n = len(y)
    m = _mask(n, mask).astype(float)
    L = laplacian(G)
    # khởi tạo bằng Tikhonov (L2)
    x = np.linalg.solve(np.diag(m) + lam * L, m * y)
    eps = eps0
    hist = [lad_objective(x, y, L, lam, mask)]
    for _ in range(max_iter):
        w = m / np.maximum(np.abs(x - y), eps)
        x = np.linalg.solve(np.diag(w) + 2 * lam * L, w * y)
        f = lad_objective(x, y, L, lam, mask)
        hist.append(f)
        rel = abs(hist[-2] - f) / max(1.0, f)
        if eps <= eps_min and rel <= tol:
            break                                   # dừng: eps nhỏ nhất và f không đổi
        if eps > eps_min and rel <= tol_stage:
            eps = max(eps * 0.1, eps_min)           # giai đoạn hiện tại đã hội tụ -> giảm eps
    return (x, np.array(hist)) if return_history else x


# ----------------------------------------------------------------------------
# Baseline L2: min ||M(x-y)||^2 + lam x^T L x  ->  (M + lam L) x = M y
# ----------------------------------------------------------------------------
def solve_tikhonov(G, y, mask=None, lam=1.0):
    m = _mask(len(y), mask).astype(float)
    L = laplacian(G)
    return np.linalg.solve(np.diag(m) + lam * L, m * np.asarray(y, float))


# ----------------------------------------------------------------------------
# Kiểm tra KKT:  0 ∈ s + 2 lam L x,  s_i = sign(x_i - y_i) nếu x_i != y_i, s_i ∈ [-1,1] nếu bằng
# Trả về max vi phạm (≈0 nếu tối ưu) và số đỉnh "dính" x_i == y_i.
# ----------------------------------------------------------------------------
def kkt_violation(G, y, x, lam, mask=None, tol=1e-5):
    m = _mask(len(y), mask)
    g = 2 * lam * laplacian(G) @ x          # cần s = -g
    r = x - y
    viol = np.zeros(len(y))
    tight = m & (np.abs(r) <= tol)
    loose = m & ~tight
    viol[~m] = np.abs(g[~m])                                   # đỉnh bị che: s=0
    viol[tight] = np.maximum(np.abs(g[tight]) - 1, 0)          # |g| <= 1
    viol[loose] = np.abs(g[loose] + np.sign(r[loose]))         # g = -sign(r)
    return float(viol.max()), int(tight.sum())

## 4. Kiểm tra tính đúng đắn

**Ngưỡng $\lambda_0$.** Điều kiện tối ưu: $0\in\partial\|x-y\|_1+2\lambda Lx$. Tại $x=y$, $\partial|0|=[-1,1]$ nên $x=y$ là nghiệm khi $|2\lambda(Ly)_i|\le1\ \forall i$, tức
$$\lambda\le\lambda_0=\frac{1}{2\|Ly\|_\infty}.$$
Dưới ngưỡng này LAD **trả lại nguyên y** ⇒ lưới λ phải bắt đầu từ $\lambda_0$.

In [ ]:
lam0 = lambda0(G, y)
print(f"lambda0 = {lam0:.4f}")
for lam in [0.9 * lam0, 1.1 * lam0]:
    print(f"  lam = {lam:.4f}:  max|x - y| = {np.abs(solve_lad(G, y, lam=lam) - y).max():.2e}")

print("\nCVXPY vs IRLS:")
print(f"{'lam':>8} {'f_cvx':>12} {'f_irls':>12} {'rel gap':>9} {'max|dx|':>9} {'KKT_cvx':>9} {'#x=y':>5}")
for lam in lam0 * np.array([1.5, 3, 10, 30, 100]):
    xc = solve_lad(G, y, lam=lam)
    xi = solve_lad_irls(G, y, lam=lam)
    fc, fi = lad_objective(xc, y, L, lam), lad_objective(xi, y, L, lam)
    kv, ntight = kkt_violation(G, y, xc, lam)
    print(f"{lam:8.4f} {fc:12.6f} {fi:12.6f} {abs(fi-fc)/fc:9.1e} {np.abs(xc-xi).max():9.1e} {kv:9.1e} {ntight:5d}")

**Nhận xét.** Hai solver độc lập cho cùng giá trị hàm mục tiêu (sai lệch tương đối rất nhỏ) và KKT ≈ 0 ⇒ nghiệm đúng.
Cột `#x=y` cho thấy tính **thưa của phần dư** — đặc trưng của L1: nhiều đỉnh được giữ đúng bằng $y_i$; λ càng lớn càng ít đỉnh "dính".

**Tính không duy nhất.** $x^\top Lx$ không phạt hướng hằng $\mathbf 1$. Nếu không đỉnh nào có $x_i=y_i$ và số phần dư dương = âm (n = 32 chẵn), thì dịch $x\to x+c\mathbf 1$ trong một khoảng nhỏ **không đổi** hàm mục tiêu (giống trung vị của số phần tử chẵn). Khi đó CVXPY và IRLS có thể trả về hai nghiệm khác nhau một hằng số nhỏ nhưng cùng giá trị tối ưu — kiểm tra dưới đây.

In [ ]:
lam = 100 * lam0
xc, xi = solve_lad(G, y, lam=lam), solve_lad_irls(G, y, lam=lam)
r = xc - y
print("#(r>0) =", (r > 1e-6).sum(), " #(r<0) =", (r < -1e-6).sum(), " #(r=0) =", (np.abs(r) <= 1e-6).sum())
d = xc - xi
print(f"xc - xi: mean = {d.mean():.2e}, std quanh mean = {d.std():.2e}  -> chủ yếu lệch một hằng số")
c = np.linspace(-0.01, 0.01, 5)
print("f(x + c1):", np.round([lad_objective(xc + ci, y, L, lam) for ci in c], 6))

## 5. Đường hội tụ IRLS

In [ ]:
lam_demo = 4 * lam0
x_ref = solve_lad(G, y, lam=lam_demo)
f_ref = lad_objective(x_ref, y, L, lam_demo)
_, hist = solve_lad_irls(G, y, lam=lam_demo, return_history=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(hist, lw=1.5); ax[0].axhline(f_ref, ls="--", c="k", lw=1, label="f* (CVXPY)")
ax[0].set(xlabel="vòng lặp", ylabel="f(x_k)", title=f"IRLS, λ = {lam_demo:.3f}"); ax[0].legend()
ax[1].semilogy(np.maximum(hist - f_ref, 1e-16), lw=1.5)
ax[1].set(xlabel="vòng lặp", ylabel="f(x_k) − f*", title="Khoảng cách tới tối ưu (log)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig4_convergence.png", dpi=150); plt.show()
print(f"{len(hist)-1} vòng lặp, f_irls − f* = {hist[-1]-f_ref:.1e} (âm nhỏ = sai số làm tròn của CVXPY)")
# MM chỉ đảm bảo giảm đơn điệu hàm làm trơn với ε cố định; khi đổi ε hàm L1 gốc có thể tăng rất nhẹ
print(f"Mức tăng lớn nhất của f giữa 2 vòng lặp: {max(np.diff(hist).max(), 0):.1e}  (chỉ xảy ra lúc đổi ε; trong mỗi giai đoạn ε cố định f giảm đơn điệu)")

## 6. Thực nghiệm giả lập (seed 0–19)

Với mỗi seed: sinh $(x, y_{syn})$ bằng `make_synthetic`; quét λ trên lưới $\lambda_0(y_{syn})\cdot[1, 10^{2.5}]$ (26 điểm, log).
* **CV-λ** (con số báo cáo chính): chọn λ bằng `cv_score` trên $y_{syn}$, **không** dùng $x$ thật.
* **Oracle-λ**: λ cho RMSE nhỏ nhất (cần $x$ thật — chỉ để tham khảo cận dưới).

So sánh với baseline $\hat x=y$ và Tikhonov (L2) chọn λ cùng cách.

In [ ]:
def lad_grid(yv, num=26):
    return lambda0(G, yv) * np.logspace(0, 2.5, num)
TIK_GRID = np.logspace(-2, 1.5, 26)

def select_lambda(solve, yv, grid):
    scores = np.array([cv_score(solve, yv, dict(lam=l)) for l in grid])
    return grid[int(np.argmin(scores))], scores

t0 = time.time()
SEEDS = range(20)
rows = []
curve_ratio = np.logspace(0, 2.5, 26)            # λ / λ0 dùng chung để vẽ đường trung bình
curves = []
for s in SEEDS:
    x_true_s, ys = make_synthetic(G, y, s)
    g = lad_grid(ys)
    lam_cv, _ = select_lambda(solve_lad, ys, g)
    rm = np.array([rmse(solve_lad(G, ys, lam=l), x_true_s) for l in g])
    curves.append(rm)
    lam_t, _ = select_lambda(solve_tikhonov, ys, TIK_GRID)
    rm_t = np.array([rmse(solve_tikhonov(G, ys, lam=l), x_true_s) for l in TIK_GRID])
    rows.append(dict(seed=s,
                     y=rmse(ys, x_true_s),
                     lad_cv=rmse(solve_lad(G, ys, lam=lam_cv), x_true_s),
                     lad_oracle=rm.min(),
                     tik_cv=rmse(solve_tikhonov(G, ys, lam=lam_t), x_true_s),
                     tik_oracle=rm_t.min(),
                     ratio_cv=lam_cv / lambda0(G, ys),
                     ratio_oracle=g[rm.argmin()] / lambda0(G, ys)))
curves = np.array(curves)
print(f"20 seed xong trong {time.time()-t0:.1f}s\n")

keys = [("y", "Không xử lý (x̂ = y)"), ("tik_cv", "Tikhonov L2, CV-λ"), ("lad_cv", "LAD + Laplacian, CV-λ"),
        ("tik_oracle", "Tikhonov L2, oracle-λ"), ("lad_oracle", "LAD + Laplacian, oracle-λ")]
SYN = {k: np.array([r[k] for r in rows]) for k, _ in keys}
print(f"{'Phương pháp':32s} {'RMSE mean ± std':>18s}")
for k, name in keys:
    print(f"{name:32s} {SYN[k].mean():8.4f} ± {SYN[k].std():.4f}")
wins = (SYN["lad_cv"] < SYN["tik_cv"]).sum()
print(f"\nLAD (CV) tốt hơn Tikhonov (CV) ở {wins}/20 seed; tốt hơn y ở {(SYN['lad_cv'] < SYN['y']).sum()}/20 seed")
ratio_cv = np.array([r["ratio_cv"] for r in rows]); ratio_or = np.array([r["ratio_oracle"] for r in rows])
print(f"λ/λ0 chọn bởi CV: median {np.median(ratio_cv):.2f};  oracle: median {np.median(ratio_or):.2f}")

## 7. Dữ liệu thật: chọn λ bằng CV và sinh `x_hat`

In [ ]:
grid_real = lad_grid(y)
lam_star, cv_real = select_lambda(solve_lad, y, grid_real)
x_hat = solve_lad(G, y, lam=lam_star)
x_hat_irls = solve_lad_irls(G, y, lam=lam_star)
kv, ntight = kkt_violation(G, y, x_hat, lam_star)
cv_best = cv_real.min()

# độ ổn định của CV theo cách chia fold
cv_other = np.array([[cv_score(solve_lad, y, dict(lam=l), seed=sd) for l in grid_real] for sd in range(1, 10)])
lam_other = grid_real[cv_other.argmin(1)]

np.save(RES_DIR / "xhat_4.npy", x_hat)
print(f"λ0 = {lambda0(G, y):.4f},  λ* (CV) = {lam_star:.4f}  (= {lam_star/lambda0(G, y):.2f} λ0),  cv_score = {cv_best:.4f}")
print(f"KKT violation = {kv:.1e},  #đỉnh x̂ = y: {ntight}/{n},  |x̂_cvx - x̂_irls|∞ = {np.abs(x_hat - x_hat_irls).max():.1e}")
print(f"λ* với fold seed 1..9: {np.round(lam_other, 3)}")
print("Đã lưu results/method_4/xhat_4.npy")
print("x_hat =", np.round(x_hat, 4))

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.semilogx(grid_real, cv_real, "o-", label="cv_score (dữ liệu thật, fold seed 0)")
for row in cv_other:
    ax1.semilogx(grid_real, row, c="tab:blue", alpha=.12, lw=1)
ax1.axvline(lam_star, c="tab:blue", ls="--", lw=1)
ax1.set(xlabel="λ", ylabel="cv_score (median |x̂ − y| trên đỉnh bị che)")
ax2 = ax1.twinx()
ax2.semilogx(curve_ratio * lambda0(G, y), curves.mean(0), "s-", c="tab:orange", ms=4,
             label="RMSE giả lập trung bình (trục λ/λ0 quy đổi)")
ax2.set_ylabel("RMSE giả lập", color="tab:orange")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper left"); ax1.grid(alpha=.3)
ax1.set_title("Đường cong chọn tham số — LAD + Laplacian")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig4_lambda_curve.png", dpi=150); plt.show()

In [ ]:
res = np.abs(y - x_hat)
vmin, vmax = min(y.min(), x_hat.min()), max(y.max(), x_hat.max())
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
for ax, val, title, cmap, vr in [
        (axes[0], y, "Quan sát y", "viridis", (vmin, vmax)),
        (axes[1], x_hat, f"x̂ LAD (λ* = {lam_star:.3f})", "viridis", (vmin, vmax)),
        (axes[2], res, "|y − x̂| (đỉnh nghi outlier)", "Reds", (0, res.max()))]:
    nx.draw_networkx_edges(G, positions, width=1.0, alpha=.3, ax=ax)
    sc = nx.draw_networkx_nodes(G, positions, node_color=val, cmap=cmap, vmin=vr[0], vmax=vr[1],
                                node_size=300, ax=ax, edgecolors="k", linewidths=.4)
    nx.draw_networkx_labels(G, positions, font_size=7, ax=ax)
    fig.colorbar(sc, ax=ax, shrink=.8); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig4_graph_y_vs_xhat.png", dpi=150); plt.show()

top = np.argsort(-res)[:5]
print("5 đỉnh bị chỉnh nhiều nhất:", [(int(i), round(float(y[i]), 3), round(float(x_hat[i]), 3)) for i in top])

## 8. Kiểm thử trên nhiều bộ đề của `public_cases.json`

Không có $x_{true}$, nhưng có thể kiểm tra **tính ổn định và đúng đắn** của pipeline trên cả 100 đồ thị:
(a) trên **dữ liệu thật** của mỗi bộ đề: λ* (CV), KKT, khớp IRLS; (b) trên **dữ liệu giả lập** của 10 bộ đề đầu × 20 seed: LAD vs Tikhonov vs y.
Các hàm hợp đồng dùng biến toàn cục `G, L, n`, nên ta tạm gán lại chúng cho từng bộ đề rồi khôi phục.

In [ ]:
_G0, _L0, _n0, _y0 = G, L, n, y
t0 = time.time()
real_rows = []
for cid in sorted(CASES):
    G, yc = case_to_graph(CASES[cid]); L = laplacian(G); n = len(yc)
    g = lad_grid(yc)
    ls, sc = select_lambda(solve_lad, yc, g)
    xc = solve_lad(G, yc, lam=ls); xi = solve_lad_irls(G, yc, lam=ls)
    kv, nt = kkt_violation(G, yc, xc, ls)
    fc, fi = lad_objective(xc, yc, L, ls), lad_objective(xi, yc, L, ls)
    real_rows.append((cid, ls / lambda0(G, yc), sc.min(), kv, abs(fi - fc) / fc, nt, np.abs(xc - yc).max()))
R = np.array([r[1:] for r in real_rows], float)
print(f"(a) 100 bộ đề, dữ liệu thật — {time.time()-t0:.1f}s")
print(f"    λ*/λ0: median {np.median(R[:,0]):.2f} [min {R[:,0].min():.2f}, max {R[:,0].max():.2f}],"
      f" số bộ đề λ* ở biên lưới: {(R[:,0] >= 10**2.5 * 0.999).sum()}")
print(f"    cv_score: median {np.median(R[:,1]):.3f};  max KKT violation {R[:,2].max():.1e};"
      f"  max gap IRLS/CVXPY {R[:,3].max():.1e}")
print(f"    #đỉnh x̂ = y: median {np.median(R[:,4]):.0f}/32;  max|x̂ − y|: median {np.median(R[:,5]):.2f}")

t0 = time.time()
syn_multi = []
for cid in sorted(CASES)[:10]:
    G, yc = case_to_graph(CASES[cid]); L = laplacian(G); n = len(yc)
    for s in range(20):
        xt, ys = make_synthetic(G, yc, s)
        ll, _ = select_lambda(solve_lad, ys, lad_grid(ys))
        lt, _ = select_lambda(solve_tikhonov, ys, TIK_GRID)
        syn_multi.append((rmse(ys, xt), rmse(solve_lad(G, ys, lam=ll), xt), rmse(solve_tikhonov(G, ys, lam=lt), xt)))
S = np.array(syn_multi)
print(f"\n(b) 10 bộ đề × 20 seed giả lập — {time.time()-t0:.1f}s")
for j, name in enumerate(["x̂ = y", "LAD + Laplacian (CV-λ)", "Tikhonov L2 (CV-λ)"]):
    print(f"    {name:26s} RMSE {S[:,j].mean():.4f} ± {S[:,j].std():.4f}")
print(f"    LAD tốt hơn Tikhonov ở {(S[:,1] < S[:,2]).mean()*100:.0f}% trường hợp, tốt hơn y ở {(S[:,1] < S[:,0]).mean()*100:.0f}%")

G, L, n, y = _G0, _L0, _n0, _y0          # khôi phục bộ đề của nhóm

### Xuất CSV đúng định dạng Mục VIII (để đưa vào notebook so sánh `evaluation/`)

In [ ]:
import csv

def save_method_csv(x_hat, filename, representative_mssv=MSSV_DAI_DIEN, group_mssvs=NHOM_MSSV):
    """Giống save_submission của đề (4 cột, UTF-8-BOM, 10 chữ số thập phân), chỉ khác tên file."""
    pred = np.asarray(x_hat, dtype=float).reshape(-1)
    assert pred.shape == y.shape and np.all(np.isfinite(pred))
    ids = [str(v).strip() for v in group_mssvs]
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["representative_mssv", "group_mssvs", "node_id", "x_hat"])
        for node_id, val in enumerate(pred):
            w.writerow([representative_mssv, "|".join(ids), node_id, f"{float(val):.10f}"])
    print(f"Đã xuất {filename} ({len(pred)} dòng)")

save_method_csv(x_hat, RES_DIR / "method_4.csv")

## 9. Dòng kết quả (dán vào bảng so sánh của nhóm)

In [ ]:
RESULT_LINE = (f"LAD + Laplacian | λ* = {lam_star:.4f} ({lam_star/lambda0(G, y):.2f}·λ0) | "
               f"RMSE_syn (CV-λ) = {SYN['lad_cv'].mean():.4f} ± {SYN['lad_cv'].std():.4f} | "
               f"cv_score = {cv_best:.4f}")
print(RESULT_LINE)
assert np.allclose(np.load(RES_DIR / "xhat_4.npy"), x_hat) and x_hat.shape == y.shape and np.all(np.isfinite(x_hat))

---
# THUYẾT MINH — Phương pháp #4: LAD + Laplacian

*(Số liệu dưới đây là của bộ đề 20260001 với MSSV mẫu; nếu đổi MSSV, đọc lại số ở các cell output phía trên.)*

### 1. Giả thiết về tín hiệu, nhiễu và các đỉnh gần nhau
* **Tín hiệu trơn trên đồ thị:** hai cảm biến nối bởi cạnh có trọng số $w_{ij}$ lớn thì có giá trị thật gần nhau ⇒ năng lượng biến thiên $x^\top Lx=\sum w_{ij}(x_i-x_j)^2$ nhỏ.
* **Nhiễu cộng, độc lập, đuôi dày:** $y_i=x_i+e_i$, ngoài nhiễu đo nhỏ còn có vài **outlier** lớn. Ta mô hình hoá $e_i\sim\text{Laplace}(0,b)$ — phân phối có đuôi dày hơn Gauss, nên khả năng xuất hiện sai số lớn không bị coi là "gần như không thể".
* Đồ thị liên thông (đề bài đảm bảo) ⇒ $\ker L=\mathrm{span}\{\mathbf 1\}$.

### 2. Hàm mục tiêu
$$\hat x=\arg\min_x\ \underbrace{\sum_{i\in\mathcal M}|x_i-y_i|}_{\text{khớp dữ liệu (L1)}}+\lambda\underbrace{x^\top Lx}_{\text{độ trơn}}$$
* **Khớp dữ liệu L1** = $-\log$ likelihood của nhiễu Laplace. Phạt *tuyến tính* theo phần dư ⇒ ảnh hưởng của một điểm bị chặn (đạo hàm $\pm1$), outlier không kéo nghiệm đi xa như phạt bình phương (đạo hàm $2r$ tăng không giới hạn).
* **Độ trơn** $x^\top Lx$ = $-\log$ tiên nghiệm GMRF $p(x)\propto e^{-\lambda x^\top Lx}$. Toàn bộ bài toán là **ước lượng MAP**.
* $\lambda$ cân bằng hai số hạng; $\mathcal M$ là tập đỉnh quan sát (dùng khi che đỉnh để CV).
* **Tính chất đặc trưng của L1 — phần dư thưa:** điều kiện tối ưu $0\in\partial\|x-y\|_1+2\lambda Lx$ cho thấy với $\lambda\le\lambda_0=1/(2\|Ly\|_\infty)$ nghiệm **chính là y** (kiểm chứng ở Mục 4: $\lambda=0.9\lambda_0$ cho $\max|x-y|\approx 10^{-8}$). Với λ lớn hơn, nhiều đỉnh vẫn giữ đúng $\hat x_i=y_i$ (21/32 đỉnh ở λ*), chỉ các đỉnh "lệch so với láng giềng" bị chỉnh ⇒ L1 tự động *chọn ra* đỉnh nghi là outlier.

### 3. Phương pháp tối ưu, lựa chọn tham số, kiểm tra
* **Dạng bài toán:** lồi, viết được thành QP: $\min \mathbf 1^\top t+\lambda\|Dx\|^2$ s.t. $-t\le x-y\le t$, $D=\sqrt W B$.
* **Solver chính:** CVXPY + Clarabel (điểm trong) — nghiệm chính xác tới ~$10^{-8}$; bài toán viết dạng DPP nên mỗi lần giải ~2 ms.
* **Solver đối chiếu tự cài — IRLS (majorize–minimize):** mỗi vòng giải $(W_k+2\lambda L)x=W_ky$ với $W_k=\mathrm{diag}(1/\max(|r_i|,\varepsilon))$; $\varepsilon$ giảm $10^{-1}\to10^{-10}$ khi mỗi giai đoạn hội tụ (thay đổi tương đối của $f<10^{-7}$); dừng khi $\varepsilon$ nhỏ nhất và thay đổi $<10^{-12}$. Hội tụ trong ~140 vòng (hình `fig4_convergence.png`, mỗi "bậc thang" là một lần giảm ε).
* **Kiểm tra đúng đắn:** (i) hai solver độc lập cho cùng $f^*$ (sai lệch tương đối $\le10^{-9}$); (ii) vi phạm KKT ≈ $10^{-8}$; (iii) chạy trên **cả 100 bộ đề** của `public_cases.json`: sai lệch IRLS/CVXPY lớn nhất $5\cdot10^{-8}$.
* **Chọn λ:** lưới log 26 điểm $\lambda\in\lambda_0\cdot[1,10^{2.5}]$ (bắt đầu từ $\lambda_0$ vì dưới đó nghiệm không đổi), chọn λ cực tiểu `cv_score` (5-fold, median |sai số| trên đỉnh bị che). Trên dữ liệu thật: **λ* = 0.223 ≈ 3.2 λ₀, cv_score = 0.300**; cực tiểu nằm *bên trong* lưới (hình `fig4_lambda_curve.png`).

### 4. Thực nghiệm, trực quan hoá và giới hạn
**Giả lập 20 seed (đồ thị của nhóm):**

| Phương pháp | RMSE (mean ± std) |
|---|---|
| Không xử lý ($\hat x=y$) | 0.293 ± 0.032 |
| Tikhonov L2, CV-λ | 0.273 ± 0.034 |
| **LAD + Laplacian, CV-λ** | **0.260 ± 0.058** |
| Tikhonov L2, oracle-λ | 0.226 ± 0.033 |
| LAD + Laplacian, oracle-λ | 0.200 ± 0.043 |

Trên 10 bộ đề × 20 seed: LAD 0.223 ± 0.066 vs Tikhonov 0.256 ± 0.057 vs y 0.275; LAD tốt hơn Tikhonov ở 71% trường hợp, tốt hơn y ở 97%.

**Trên dữ liệu thật** (`fig4_graph_y_vs_xhat.png`): các đỉnh bị chỉnh mạnh nhất là 28 (17.55 → 18.77), 3, 12, 29, 20 — đều là đỉnh có giá trị khác hẳn láng giềng trên hình y; 21/32 đỉnh giữ nguyên.

**Giới hạn:**
1. **Chọn λ là khâu yếu nhất.** CV chọn λ nhỏ hơn oracle (median 1.8 λ₀ so với 2.8 λ₀) và dao động theo cách chia fold (λ* từ 0.14 đến 1.12 với fold seed 1–9) vì chỉ có 32 đỉnh ⇒ khoảng cách giữa CV-λ (0.260) và oracle-λ (0.200) còn lớn. Có thể cải thiện bằng CV lặp nhiều lần hoặc lấy trung vị λ* theo nhiều fold seed.
2. **Nghiệm có thể không duy nhất:** khi không đỉnh nào có $\hat x_i=y_i$ và số phần dư dương = âm, dịch $x+c\mathbf 1$ không đổi $f$ (đã minh hoạ ở Mục 4, xảy ra với λ rất lớn; không xảy ra ở λ*).
3. **Kém hiệu quả với nhiễu Gauss nhỏ:** L1 tương ứng nhiễu Laplace; với phần nhiễu Gauss "bình thường", ước lượng bình phương/Huber hiệu quả hơn. Phần dư thưa cũng nghĩa là nhiều đỉnh **giữ nguyên nhiễu** của y.
4. $x^\top Lx$ làm mờ các bước nhảy thật của tín hiệu (nếu có) — Graph TV (#5) phù hợp hơn cho tín hiệu từng mảnh hằng.
5. Đồ thị một bộ đề (1/100) cho λ* chạm biên trên của lưới — cần mở rộng lưới nếu nhóm rơi vào bộ đề đó.